# S&P 500 Market Predictor

**Created by**: Eric Liu, Rithikesh Muddana



## Imports

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


Device: cpu


# DATA LOADING AND FEATURE ENGINEERING

In this section, all data files are merged together into one DataFrame. Luckily, all the data except for unemployment_rate followed the same date pattern (weekdays only) and were not missing any data points, so it was relatively simple to merge the data. Unemployment date was measured by month, however, so we filled in the daily unemployment rates by taking the most recent monthly value and applying it to all days in that month. Outliers were capped using winsorsizing, essnentially a technique that caps outliers at a specific percentile. Financial outliers are genuinely informative, so it doesn't make sense to outright discard them.

As for feature engineering, we use the existing data to extrapolate new features to use in training. The features are listed and explained below:

### Returns + Lags

`Return_1d / 3d / 5d / 10d / 20d`

Percentage change in closing price over the last 1, 3, 5, 10, and 20 days. These capture short and medium-term momentum — the idea being that recent price direction may persist or change

`Lag_1 through Lag_5`

Yesterday's return, the day before's return, and so on up to 5 days back. These give the model a "memory" of recent daily moves as individual inputs rather than just a rolling sum.

`Vol_5d`

Standard deviation of daily returns over the last 5 days. Measures how choppy the market has been recently.

### Volatility

`Vol_20d`

Same but over 20 days — a smoother, more stable estimate.

`Vol_ratio`

Vol_5d / Vol_20d. When this is above 1, short-term volatility is elevated relative to the recent norm — often signals turbulence from an event.

### Moving Averages & Crossovers

`Close_vs_SMA5/10/20/50/200`

How far the current price sits above or below each moving average, expressed as a percentage. A positive value means price is above the average (bullish), negative means below (bearish). The 200-day in particular is widely watched by institutional traders.

`SMA5_vs_SMA20`

Short-term average relative to medium-term. When the fast MA crosses above the slow MA, it's a classic "golden cross" — a bullish signal.

`SMA20_vs_SMA50`

Medium-term vs intermediate-term trend alignment.

`SMA50_vs_SMA200`

The big-picture trend. When the 50-day is above the 200-day, the market is `nerally in a long-term uptrend.

### RSI

`RSI_14`

Relative Strength Index over 14 days. Ranges from 0 to 100. Above 70 is traditionally "overbought" (potential reversal down), below 30 is "oversold" (potential bounce). It measures the speed and magnitude of recent price moves.

### MACD

`MACD`

The difference between the 12-day and 26-day exponential moving averages. Positive means the short-term trend is stronger than the long-term trend.

`MACD_signal`

A 9-day EMA smoothing of the MACD line itself — acts as a trigger line.

`MACD_hist MACD - MACD_signal.`

When positive and growing, momentum is accelerating upward. When it crosses zero, it's a buy/sell signal in classic technical analysis.

### Bollinger Bands

`BB_pct`

Where the current price sits within the Bollinger Band, from 0 (at the lower band) to 1 (at the upper band). Above 1 or below 0 means the price has broken out of the bands entirely.

`BB_width`

How wide the bands are relative to the moving average — a measure of volatility. Narrow bands (low width) often precede large moves.

In [42]:
import glob
import os
import pandas as pd

def load_data() -> pd.DataFrame:

    # --- Load S&P 500 data (index_val files) ---
    index_val_files = glob.glob('index_val*.csv')

    all_dfs = []

    for file in index_val_files:
        df = pd.read_csv(file)
        all_dfs.append(df)

    # Concatenate all dataframes into one
    combined_df = pd.concat(all_dfs, ignore_index=True)

    # Convert the 'Date' column to datetime
    combined_df['Date'] = pd.to_datetime(combined_df['Date'])

    # Convert OHLC columns to numeric
    price_cols = ['Open', 'High', 'Low', 'Close']
    combined_df[price_cols] = (
        combined_df[price_cols]
        .apply(lambda col: col.astype(str).str.replace(r'[\$,\s]', '', regex=True))
        .apply(pd.to_numeric, errors='coerce')
    )

    # Helper to standardize date column (because in some datasets, data is called observation_date, or date)
    def _standardize_date_column(df: pd.DataFrame, file_name: str) -> pd.DataFrame:
        date_col = None
        for col in df.columns:
            if col.lower() == 'date':
                date_col = col
                break
            if col.lower() == 'observation_date':
                date_col = col
                break

        if date_col:
            if date_col != 'Date':
                df = df.rename(columns={date_col: 'Date'})
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
            return df
        else:
            print(f"Warning: '{file_name}' does not contain a 'date' or 'observation_date' column. Skipping this file.")
            return None # Return None if date column not found

    # --- treasury_yield.csv ---
    treasury_df = pd.read_csv('treasury_yield.csv')
    treasury_df = _standardize_date_column(treasury_df, 'treasury_yield.csv')

    yield_cols = [col for col in treasury_df.columns if col.lower() != 'date']
    treasury_df = treasury_df[['Date', yield_cols[0]]]
    treasury_df = treasury_df.rename(columns={yield_cols[0]: 'Treasury_Yield'})
    treasury_df['Treasury_Yield'] = pd.to_numeric(
        treasury_df['Treasury_Yield'].astype(str).str.replace(r'[\%,]', '', regex=True),
        errors='coerce'
    )
    combined_df = pd.merge(combined_df, treasury_df, on='Date', how='left')


    # --- unemployment_rate.csv ---
    unemployment_df = pd.read_csv('unemployment_rate.csv')
    unemployment_df = _standardize_date_column(unemployment_df, 'unemployment_rate.csv')

    unemp_cols = [col for col in unemployment_df.columns if col.lower() != 'date']
    unemployment_df = unemployment_df[['Date', unemp_cols[0]]]
    unemployment_df = unemployment_df.rename(columns={unemp_cols[0]: 'Unemployment_Rate'})
    unemployment_df['Unemployment_Rate'] = pd.to_numeric(
        unemployment_df['Unemployment_Rate'].astype(str).str.replace(r'[\%,]', '', regex=True),
        errors='coerce'
    )
    combined_df = pd.merge(combined_df, unemployment_df, on='Date', how='left')


    # --- vix.csv ---
    vix_df = pd.read_csv('vix.csv')
    vix_df = _standardize_date_column(vix_df, 'vix.csv')

    vix_df = vix_df[['Date', 'VIXCLS']].rename(columns={'VIXCLS': 'VIX_Close'})
    vix_df['VIX_Close'] = pd.to_numeric(
        vix_df['VIX_Close'].astype(str).str.replace(r'[\$,\s]', '', regex=True),
        errors='coerce'
    )
    combined_df = pd.merge(combined_df, vix_df, on='Date', how='left')

    # --- Apply cutoff date, sort by date ---
    combined_df = combined_df[combined_df['Date'] < '2026-01-01'].reset_index(drop=True)
    combined_df = combined_df.sort_values(by='Date').reset_index(drop=True)

    return combined_df

def data_cleaning(df):
    # Work on a copy — avoid mutating the original df
    df = df.copy()

    # Unemployment rate is monthly, forward-fill across trading days
    df['Unemployment_Rate'] = df['Unemployment_Rate'].ffill().bfill()

    # Treasury yield has 7 missing values — forward-fill then back-fill
    df['Treasury_Yield'] = df['Treasury_Yield'].ffill().bfill()

    # Winsorize all feature columns at 1st/99th percentile (winsorize is essentially making outliers capped to a specific percentile)
    feature_cols = [col for col in df.columns if col.lower() != 'date']
    for col in feature_cols:
        lower = df[col].quantile(0.01)
        upper = df[col].quantile(0.99)
        df[col] = df[col].clip(lower, upper)

    return df

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    # --- Target: 1 if next day closes higher, else 0 ---
    d["Target"] = (d["Close"].shift(-1) > d["Close"]).astype(int)

     # --- Returns & lags ---
    d["Return_1d"]  = d["Close"].pct_change(1)
    d["Return_3d"]  = d["Close"].pct_change(3)
    d["Return_5d"]  = d["Close"].pct_change(5)
    d["Return_10d"] = d["Close"].pct_change(10)
    d["Return_20d"] = d["Close"].pct_change(20)

    for lag in [1, 2, 3, 4, 5]:
        d[f"Lag_{lag}"] = d["Return_1d"].shift(lag)

    # --- Volatility ---
    d["Vol_5d"]  = d["Return_1d"].rolling(5).std()
    d["Vol_20d"] = d["Return_1d"].rolling(20).std()
    d["Vol_ratio"] = d["Vol_5d"] / d["Vol_20d"]

    # --- Moving averages & crossovers ---
    for w in [5, 10, 20, 50, 200]:
        d[f"SMA_{w}"]       = d["Close"].rolling(w).mean()
        d[f"Close_vs_SMA{w}"] = d["Close"] / d[f"SMA_{w}"] - 1

    d["SMA5_vs_SMA20"]  = d["SMA_5"]  / d["SMA_20"]  - 1
    d["SMA20_vs_SMA50"] = d["SMA_20"] / d["SMA_50"]  - 1
    d["SMA50_vs_SMA200"] = d["SMA_50"] / d["SMA_200"] - 1

    # --- RSI ---
    delta = d["Close"].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / loss.replace(0, np.nan)
    d["RSI_14"] = 100 - (100 / (1 + rs))

    # --- MACD ---
    ema12 = d["Close"].ewm(span=12, adjust=False).mean()
    ema26 = d["Close"].ewm(span=26, adjust=False).mean()
    d["MACD"]        = ema12 - ema26
    d["MACD_signal"] = d["MACD"].ewm(span=9, adjust=False).mean()
    d["MACD_hist"]   = d["MACD"] - d["MACD_signal"]

    # --- Bollinger Bands ---
    sma20  = d["Close"].rolling(20).mean()
    std20  = d["Close"].rolling(20).std()
    d["BB_upper"] = sma20 + 2 * std20
    d["BB_lower"] = sma20 - 2 * std20
    d["BB_pct"]   = (d["Close"] - d["BB_lower"]) / (d["BB_upper"] - d["BB_lower"])
    d["BB_width"] = (d["BB_upper"] - d["BB_lower"]) / sma20

    # --- Intraday range features ---
    d["High_Low_range"]  = (d["High"] - d["Low"]) / d["Close"]
    d["Open_Close_gap"]  = (d["Open"] - d["Close"].shift(1)) / d["Close"].shift(1)

    return d

df = load_data()
df = build_features(df)
df = data_cleaning(df)

display(df.head())
display(df.info())
display(df.describe())

print("Dataset:", df.shape)


,Date,Open,High,Low,Close,Treasury_Yield,Unemployment_Rate,VIX_Close,Target,Return_1d,...,RSI_14,MACD,MACD_signal,MACD_hist,BB_upper,BB_lower,BB_pct,BB_width,High_Low_range,Open_Close_gap
0,2022-01-18,4632.24,4632.24,4568.70,4577.11,1.87,3.9,22.79,0,NaN,...,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.013882,NaN
1,2022-01-19,4588.03,4611.55,4530.20,4532.76,1.83,3.9,23.85,0,-0.009690,...,NaN,-3.537892,-0.707578,-2.830313,NaN,NaN,NaN,NaN,0.017947,0.002386
2,2022-01-20,4547.35,4602.11,4477.95,4482.73,1.83,3.9,25.59,0,-0.011037,...,NaN,-10.260424,-2.618147,-7.642276,NaN,NaN,NaN,NaN,0.027697,0.003219
3,2022-01-21,4471.38,4494.52,4395.34,4397.94,1.81,3.9,28.85,1,-0.018915,...,NaN,-22.174311,-6.529380,-15.644931,NaN,NaN,NaN,NaN,0.022551,-0.002532
4,2022-01-24,4356.32,4417.35,4222.62,4410.13,1.81,3.9,29.90,0,0.002772,...,NaN,-30.283433,-11.280191,-19.003242,NaN,NaN,NaN,NaN,0.036918,-0.009464


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 993 entries, 0 to 992
Data columns (total 45 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               993 non-null    datetime64[ns]
 1   Open               993 non-null    float64       
 2   High               993 non-null    float64       
 3   Low                993 non-null    float64       
 4   Close              993 non-null    float64       
 5   Treasury_Yield     993 non-null    float64       
 6   Unemployment_Rate  993 non-null    float64       
 7   VIX_Close          993 non-null    float64       
 8   Target             993 non-null    int64         
 9   Return_1d          992 non-null    float64       
 10  Return_3d          990 non-null    float64       
 11  Return_5d          988 non-null    float64       
 12  Return_10d         983 non-null    float64       
 13  Return_20d         973 non-null    float64       
 14  Lag_1     

None

,Date,Open,High,Low,Close,Treasury_Yield,Unemployment_Rate,VIX_Close,Target,Return_1d,...,RSI_14,MACD,MACD_signal,MACD_hist,BB_upper,BB_lower,BB_pct,BB_width,High_Low_range,Open_Close_gap
count,993,993.000000,993.000000,993.000000,993.000000,993.000000,993.000000,993.000000,993.000000,992.000000,...,979.000000,993.000000,993.000000,993.000000,974.000000,974.000000,974.000000,974.000000,993.000000,992.000000
mean,2024-01-08 16:49:18.308157184,5009.362790,5037.661313,4978.664878,5009.970224,3.876113,3.893958,19.208848,0.531722,0.000425,...,55.885224,15.877312,15.741150,0.136569,5166.030087,4827.871823,0.596328,0.069829,0.012064,0.000249
min,2022-01-18 00:00:00,3681.374400,3711.656800,3635.783600,3666.690800,1.810000,3.500000,12.217600,0.000000,-0.032080,...,21.295785,-118.682116,-113.653810,-38.594180,3900.269977,3522.724661,-0.104457,0.027072,0.003082,-0.015861
25%,2023-01-12 00:00:00,4155.710000,4177.510000,4127.180000,4155.170000,3.610000,3.600000,14.910000,0.000000,-0.005026,...,42.637582,-17.997750,-16.011613,-9.706224,4325.646507,4050.235223,0.358504,0.044739,0.007003,-0.002181
50%,2024-01-09 00:00:00,4764.730000,4785.390000,4747.120000,4774.750000,4.090000,3.900000,17.800000,1.000000,0.000693,...,57.477057,29.228684,26.698028,0.292100,4869.195893,4679.002605,0.683044,0.061492,0.010273,0.000443
75%,2025-01-03 00:00:00,5817.800000,5846.520000,5786.080000,5815.260000,4.310000,4.200000,22.290000,1.000000,0.006300,...,67.978102,55.925250,53.217859,9.880968,6032.814640,5646.875657,0.849288,0.085378,0.014958,0.002908
max,2025-12-31 00:00:00,6875.517600,6899.915600,6843.988800,6875.426400,4.830000,4.400000,33.836000,1.000000,0.025983,...,88.815753,96.496614,91.555704,44.139596,6967.048518,6655.275063,1.080474,0.177469,0.036918,0.014222
std,NaN,934.997524,934.234596,935.075930,934.938520,0.660657,0.300342,5.418499,0.499244,0.010346,...,16.044455,50.488397,47.593983,15.932419,920.720123,925.430733,0.317246,0.033625,0.007012,0.004815


Dataset: (993, 45)
